# Plant Disease Detection — Brinjal & Tomato (Colab)

**What this notebook does:** downloads **two open datasets for brinjal (eggplant) and tomato** (654 MB in total), merges them into **one 20-class problem**, fine-tunes **one EfficientNet-B0**, and saves a **~21 MB** weight file.

Run each cell in order, or **Runtime → Run all**. A GPU is recommended but **not required**.

| Item | Value |
|---|---|
| Dataset 1 | `Project-AgML/eggplant_leaf_disease_classification` (config `raw`) — brinjal, 10 classes, 3,116 images |
| Dataset 2 | `Project-AgML/tomato_leaf_disease` — tomato, 10 classes, 11,000 images |
| Total images | 14,116 (80 % train / 20 % validation) |
| Classes | 20 — every label is stored as `Crop___Disease` |
| Crops | Brinjal (eggplant) and Tomato |
| Model | EfficientNet-B0, ImageNet weights |
| Checkpoint | ~21 MB `plant_disease_model.pth` |
| Time | ~6–10 min on a free Colab T4 |

Class names are read **from the datasets themselves** and saved inside the checkpoint, so the backend can never fall out of sync with the model.

> **Why only two crops?** There is no small public dataset with disease labels for ladies finger (okra) or cucumber — the only okra dataset on the Hub is 8.4 GB and the cucumber ones are 2.1–2.6 GB. Brinjal and tomato have compact, well-labelled alternatives, so those two are used.


In [ ]:
# ── Cell 1: install / imports / settings ─────────────────────────────────────
import importlib, subprocess, sys, random, json, datetime
from pathlib import Path

def _ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("datasets")
_ensure("scikit-learn", "sklearn")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import (load_dataset, concatenate_datasets, ClassLabel,
                      Features, Image as HFImage, Value)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

SEED          = 42
IMAGE_SIZE    = 224
BATCH_SIZE    = 32
EPOCHS        = 4
LEARNING_RATE = 3e-4
VAL_SPLIT     = 0.2
NUM_WORKERS   = 2
CKPT_PATH     = Path("/content/plant_disease_model.pth")

# (Hugging Face dataset id, config or None, crop name used as the label prefix)
SOURCES = [
    ("Project-AgML/eggplant_leaf_disease_classification", "raw", "Brinjal"),
    ("Project-AgML/tomato_leaf_disease", None, "Tomato"),
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)
print("PyTorch:", torch.__version__)
if device.type == "cpu":
    print("No GPU found - training still works, it is just slower.")


In [ ]:
# ── Cell 2: download both vegetable datasets and merge them ─────────────────
loaded = []
for path, config, crop in SOURCES:
    split = load_dataset(path, config, split="train") if config else load_dataset(path, split="train")
    loaded.append((crop, split))
    print(f"{crop:8s} {path:55s} {len(split):6,} images, "
          f"{len(split.features['label'].names)} classes")

# Build ONE shared label space: "Crop___Disease"
CLASS_NAMES = []
for crop, split in loaded:
    for name in split.features["label"].names:
        key = f"{crop}___{name}"
        if key not in CLASS_NAMES:
            CLASS_NAMES.append(key)

NUM_CLASSES = len(CLASS_NAMES)
index_of = {name: i for i, name in enumerate(CLASS_NAMES)}

parts = []
for crop, split in loaded:
    lookup = {i: index_of[f"{crop}___{n}"]
              for i, n in enumerate(split.features["label"].names)}
    part = split.select_columns(["image", "label"])
    part = part.add_column("label_id", [lookup[i] for i in split["label"]])
    part = part.remove_columns(["label"]).rename_column("label_id", "label")
    # Force both parts to the exact same schema so they can be concatenated.
    part = part.cast(Features({"image": HFImage(), "label": Value("int64")}))
    parts.append(part)

data = concatenate_datasets(parts).cast_column("label", ClassLabel(names=CLASS_NAMES))

print(f"\nTotal images : {len(data):,}")
print(f"Classes      : {NUM_CLASSES}")
print(f"Crops        : {', '.join(sorted({c for c, _ in loaded}))}\n")

counts = np.bincount(np.asarray(data["label"], dtype=np.int64), minlength=NUM_CLASSES)
for i, name in enumerate(CLASS_NAMES):
    print(f"  {i:2d}. {name:42s} {counts[i]:5,} images")


In [ ]:
# ── Cell 3: train/validation split, transforms and data loaders ─────────────
split = data.train_test_split(test_size=VAL_SPLIT, seed=SEED, stratify_by_column="label")
train_ds, val_ds = split["train"], split["test"]
print(f"Train      : {len(train_ds):,} images")
print(f"Validation : {len(val_ds):,} images")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class LeafDataset(Dataset):
    """Wraps a Hugging Face split and returns (tensor, label)."""

    def __init__(self, hf_split, transform):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, index):
        row = self.hf_split[index]
        return self.transform(row["image"].convert("RGB")), row["label"]


train_loader = DataLoader(
    LeafDataset(train_ds, train_tf),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    LeafDataset(val_ds, val_tf),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))


In [ ]:
# ── Cell 4: the model (this exact function is copied in the backend) ────────
def build_model(num_classes):
    """EfficientNet-B0 + small custom head. ~5.3M params, ~21 MB checkpoint."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes),
    )
    return model


model = build_model(NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

criterion  = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler     = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")


In [ ]:
# ── Cell 5: training loop (saves the best epoch) ────────────────────────────
def run_epoch(loader, training):
    model.train() if training else model.eval()
    loss_sum, correct, seen = 0.0, 0, 0

    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, targets in tqdm(loader, leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            loss_sum += loss.item() * targets.size(0)
            correct  += (logits.argmax(1) == targets).sum().item()
            seen     += targets.size(0)

    return loss_sum / seen, 100.0 * correct / seen


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, training=True)
    vl_loss, vl_acc = run_epoch(val_loader, training=False)
    scheduler.step()

    history["train_loss"].append(round(tr_loss, 4))
    history["train_acc"].append(round(tr_acc, 2))
    history["val_loss"].append(round(vl_loss, 4))
    history["val_acc"].append(round(vl_acc, 2))

    print(f"Epoch {epoch}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.2f}% "
          f"| val loss {vl_loss:.4f} acc {vl_acc:.2f}%")

    if vl_acc > best_acc:
        best_acc = vl_acc
        torch.save({
            "architecture": "efficientnet_b0",
            "image_size": IMAGE_SIZE,
            "num_classes": NUM_CLASSES,
            "class_names": list(CLASS_NAMES),
            "model_state_dict": model.state_dict(),
            "val_acc": round(vl_acc, 2),
            "best_epoch": epoch,
            "history": history,
            "dataset": "Brinjal + Tomato (Project-AgML)",
            "train_images": len(train_ds),
            "val_images": len(val_ds),
        }, CKPT_PATH)
        print(f"  saved best checkpoint (val acc {vl_acc:.2f}%)")

print(f"\nBest validation accuracy: {best_acc:.2f}%")


In [ ]:
# ── Cell 6: final evaluation on the held-out validation split ───────────────
checkpoint = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

true_labels, predicted = [], []
with torch.inference_mode():
    for images, targets in tqdm(val_loader, desc="Evaluating"):
        images = images.to(device)
        logits = model(images)
        predicted.extend(logits.argmax(1).cpu().tolist())
        true_labels.extend(targets.tolist())

accuracy = accuracy_score(true_labels, predicted) * 100
macro_f1 = f1_score(true_labels, predicted, average="macro", zero_division=0)

print(f"Test accuracy : {accuracy:.2f}%")
print(f"Macro F1      : {macro_f1:.4f}")
print()
print(classification_report(true_labels, predicted,
                            labels=list(range(NUM_CLASSES)),
                            target_names=CLASS_NAMES, zero_division=0,
                            digits=2))


In [ ]:
# ── Cell 7: save final metrics into the checkpoint + verify it loads ────────
checkpoint["test_accuracy"] = round(accuracy, 2)
checkpoint["macro_f1"] = round(macro_f1, 4)
checkpoint["trained_at"] = datetime.datetime.now().isoformat(timespec="seconds")
torch.save(checkpoint, CKPT_PATH)

size_mb = CKPT_PATH.stat().st_size / (1024 ** 2)
print(f"Checkpoint : {CKPT_PATH}")
print(f"Size       : {size_mb:.1f} MB")
print(f"Classes    : {len(checkpoint['class_names'])}")

# Reload into a fresh model exactly like the backend does (strict=True).
fresh = build_model(checkpoint["num_classes"])
fresh.load_state_dict(torch.load(CKPT_PATH, map_location="cpu", weights_only=False)["model_state_dict"], strict=True)
print("Verified: checkpoint loads cleanly into a fresh EfficientNet-B0.")

print("\nClass list (put this table in your report):")
for i, name in enumerate(checkpoint["class_names"]):
    crop, disease = name.split("___", 1)
    print(f"  {i:2d}. {crop:8s} | {disease}")


In [ ]:
# ── Cell 8: download the weights to your PC ─────────────────────────────────
try:
    from google.colab import files
    files.download(str(CKPT_PATH))
except ImportError:
    print("Not running in Colab - your file is at:", CKPT_PATH)


## Download and use the weights

1. The cell above triggers a browser download of `plant_disease_model.pth`.
2. Save it as **`C:\CSP project\models\plant_disease_model.pth`** — that is the exact path the backend looks for.
3. Install the backend dependencies once:

```bash
pip install flask flask-cors pillow torch torchvision
```

4. Start the app:

```bash
cd "C:\CSP project"
python plant_disease_backend.py
```

5. Open **http://127.0.0.1:5000** and upload a leaf photo.

The backend reads `class_names` **from the checkpoint**, so the model and the app can never disagree about the 20 classes.
